# Fine-Tuning a Self-Hosted Mistral for Grounded, Refusal-Aware Answers

This notebook documents an additional optimisation experiment for the generation stage of the
retrieval-augmented pipeline. The generator comparison in Section 4.9 exposed a trade-off between
fluency and faithfulness. This experiment examines whether a locally hosted Mistral-7B,
instruction-tuned on the project's held-out data, moves towards the safer region of that trade-off:
answering strictly from the retrieved German AWMF context and returning an explicit refusal when
the answer is absent.

The setup reuses the components of notebook 22 without change: the AWMF chunks are read from the
Neon vector store (collection awmf_baseline_bge), retrieval is the same hybrid pipeline (dense +
BM25, fused with Reciprocal Rank Fusion, reranked by the cross-encoder, tightened to the strongest
chunks), and the reference answers for evaluation come from the independent corpus-grounded ground
truth produced in notebook 22.

Training uses only the questions of the 715-row benchmark that lie outside the 200-question
evaluation subset, so the evaluation set remains untouched. Within that held-out remainder, the
questions that map to the ten target conditions provide the answerable examples (their extracted
MedQA answer text), and the out-of-scope remainder provides the refusal examples. The generator
runs locally and spends no external calls; only the final RAGAS scoring uses the judge model. The
result is not assumed in advance, and the fine-tuned model is measured against the unmodified base
model under identical conditions.

## 1. Environment

The notebook targets a Colab GPU runtime. Two Colab secrets are required, as in notebook 22:
NEON_DATABASE_URL for the read-only vector store, and OPENROUTER_API_KEY for the RAGAS judge only.

In [ ]:
%%capture
# Retrieval stack (identical to notebook 22).
!pip install -q ragas langchain langchain-openai langchain-huggingface psycopg2-binary pgvector langchain-postgres sentence-transformers rank_bm25 sqlalchemy nest_asyncio datasets
# QLoRA training stack.
!pip install -q -U unsloth trl peft accelerate bitsandbytes

## 2. Configuration

File names, column names, retrieval knobs, and model identifiers are gathered here. The retrieval
knobs match notebook 22. The benchmark files are located on Google Drive by name.

In [ ]:
CONFIG = {
    # files (located on Drive by name)
    "BENCH_715":   "GP_Top10_Bilingual_Open_EndedQ.csv",           # full 715-row bilingual benchmark
    "EVAL_200":    None,                                           # None: prefer the AI-filtered set, else the final set
    "GT_FILE":     "AWMF_CorpusGrounded_GT_Independent.csv",       # independent corpus-grounded reference (notebook 22)

    # benchmark columns
    "Q_EN":       "English_Open_Question",
    "Q_DE":       "German_Open_Question",
    "Q_FULL_EN":  "English_Question",     # full MCQ text, used to extract the answer letter
    "ANS_LETTER": "Correct_Answer",
    "GT_Q":       "English_Open_Question",
    "GT_REF":     "corpus_ground_truth",

    # vector store and retrieval (as notebook 22)
    "COLLECTION": "awmf_baseline_bge",
    "EMBEDDER":   "BAAI/bge-m3",
    "RERANKER":   "BAAI/bge-reranker-v2-m3",
    "K_DENSE": 50, "K_BM25": 50, "RRF_K": 60, "K_FINAL": 8,

    # model and training
    "MODEL":    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "MAX_SEQ":  2048,
    "EPOCHS":   2,
    "NEG_FRACTION": 0.5,     # refusal examples as a fraction of the answerable examples
    "EVAL_N":   60,          # number of evaluation questions to score
    "OUTPUT_DIR": "mistral_grounded_lora",

    # RAGAS judge (external API; used only for scoring)
    "JUDGE_MODEL": "anthropic/claude-sonnet-4.6",   # gpt-4o-mini is cheaper while iterating
}

## 3. Benchmark, held-out split, disease filter, and answer extraction

The held-out training pool is the benchmark minus the 200 evaluation questions, matched on the
English open question text. The disease mapping and the answer-letter extraction reproduce the
logic of notebook 00. Questions that map to one of the ten target conditions form the answerable
pool; the out-of-scope remainder forms the refusal pool.

In [ ]:
import glob, re
import pandas as pd
from google.colab import drive, userdata
drive.mount('/content/drive')

def find(name):
    hits = glob.glob(f"/content/drive/MyDrive/**/{name}", recursive=True)
    return hits[0] if hits else name

bench = pd.read_csv(find(CONFIG["BENCH_715"]))
eval_path = None
for cand in (CONFIG["EVAL_200"], "AWMF_Golden_Dataset_200Q_AIFiltered.csv", "AWMF_Golden_Dataset_200Q_Final.csv"):
    if cand:
        h = glob.glob(f"/content/drive/MyDrive/**/{cand}", recursive=True)
        if h:
            eval_path = h[0]; break
eval200 = pd.read_csv(eval_path)
print("715 rows:", len(bench), "| 200 rows:", len(eval200))
print("715 columns:", list(bench.columns))

Q_EN, Q_DE = CONFIG["Q_EN"], CONFIG["Q_DE"]
eval_keys = set(eval200[Q_EN].astype(str).str.strip())
held = bench[~bench[Q_EN].astype(str).str.strip().isin(eval_keys)].copy()
print("held-out for training (715 - 200):", len(held))

In [ ]:
# Disease mapping (notebook 00).
target_diseases = [
    "Hypertension", "Diabetes", "Asthma", "COPD", "Coronary Heart Disease",
    "Heart Failure", "Chronic Back Pain", "Depression", "Osteoarthritis", "Dementia",
]
disease_keywords = {
    "Hypertension": ['hypertension', 'hypertensive', 'blood pressure is 180', 'blood pressure is 160', 'blood pressure is 170'],
    "Diabetes": ['diabetes', 'hba1c', 'diabetic'],
    "Asthma": ['asthma', 'asthmatic'],
    "COPD": ['copd', 'chronic obstructive', 'emphysema'],
    "Coronary Heart Disease": ['coronary', 'myocardial infarction', 'angina'],
    "Heart Failure": ['heart failure', 'chf', 'cardiac failure', 'heart failing'],
    "Chronic Back Pain": ['back pain', 'lumbar', 'sciatica'],
    "Depression": ['depression', 'depressed', 'major depressive'],
    "Osteoarthritis": ['osteoarthritis', 'joint pain'],
    "Dementia": ['dementia', 'alzheimer', 'memory loss', 'cognitive decline', 'forgetful'],
}
def get_disease(text):
    text = str(text).lower()
    for disease, keywords in disease_keywords.items():
        for kw in keywords:
            if kw in text:
                return disease
    return "Other"

# Answer-letter extraction (notebook 00).
def extract_answer_text(question, correct_letter):
    if pd.isna(question) or pd.isna(correct_letter):
        return ""
    pattern = rf"\({correct_letter}\)\s*(.*?)(?=\n\s*\([A-Z]\)|\n\s*\*\*Answer|\Z)"
    m = re.search(pattern, str(question), flags=re.DOTALL | re.IGNORECASE)
    if m:
        return m.group(1).strip().strip('"')
    try:
        parts = str(question).split(f"({correct_letter})")
        if len(parts) > 1:
            return parts[1].split("\n")[0].strip().strip('"')
    except Exception:
        pass
    return ""

held["Disease"] = held[Q_EN].apply(get_disease)
held["English_Correct_Text"] = held.apply(
    lambda r: extract_answer_text(r[CONFIG["Q_FULL_EN"]], r[CONFIG["ANS_LETTER"]]), axis=1)

df_pos = held[(held["Disease"] != "Other") & (held["English_Correct_Text"].astype(str).str.strip() != "")].copy()
df_neg = held[held["Disease"] == "Other"].copy()
print("answerable (in-scope) examples:", len(df_pos), "| out-of-scope pool:", len(df_neg))

## 4. Hybrid retrieval over the Neon vector store (notebook 22)

The dense store, the BM25 index, the cross-encoder reranker, and the fusion function are taken
from notebook 22. The German open question is used as the search query, which keeps this step
offline and matches the German query that notebook 22 uses for its ground-truth retrieval.

In [ ]:
import os, torch
from sqlalchemy import create_engine, text
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_postgres import PGVector
from sentence_transformers import CrossEncoder
from rank_bm25 import BM25Okapi

NEON = userdata.get("NEON_DATABASE_URL")
COLLECTION = CONFIG["COLLECTION"]

bge = HuggingFaceEmbeddings(model_name=CONFIG["EMBEDDER"], model_kwargs={"device": "cpu"})
vector_store = PGVector(embeddings=bge, collection_name=COLLECTION, connection=NEON, use_jsonb=True)

engine = create_engine(NEON, pool_pre_ping=True)
with engine.connect() as conn:
    all_texts = [r[0] for r in conn.execute(text(
        "SELECT document FROM langchain_pg_embedding "
        "WHERE collection_id = (SELECT uuid FROM langchain_pg_collection WHERE name = :c)"), {"c": COLLECTION})]
bm25_index = BM25Okapi([t.lower().split(" ") for t in all_texts])
print("chunks pulled for BM25:", len(all_texts))

_device = "cuda" if torch.cuda.is_available() else "cpu"
reranker = CrossEncoder(CONFIG["RERANKER"], max_length=512, device=_device)

K_DENSE, K_BM25, RRF_K, K_FINAL = CONFIG["K_DENSE"], CONFIG["K_BM25"], CONFIG["RRF_K"], CONFIG["K_FINAL"]

def rrf_merge(*lists, k=RRF_K):
    scores, keep = {}, {}
    for lst in lists:
        for rank, t in enumerate(lst):
            key = t[:120]
            scores[key] = scores.get(key, 0.0) + 1.0 / (k + rank + 1)
            keep.setdefault(key, t)
    return [keep[key] for key in sorted(keep, key=lambda kk: scores[kk], reverse=True)]

def hybrid_retrieve(german_query, k_final=K_FINAL):
    dense  = [d.page_content for d in vector_store.as_retriever(search_kwargs={"k": K_DENSE}).invoke(german_query)]
    sparse = bm25_index.get_top_n(german_query.lower().split(" "), all_texts, n=K_BM25)
    fused  = rrf_merge(dense, sparse)
    scores = reranker.predict([[german_query, t] for t in fused])
    return [t for _, t in sorted(zip(scores, fused), key=lambda x: x[0], reverse=True)][:k_final]

## 5. Instruction dataset

Each answerable question is paired with the context retrieved for its German form, and the target
is the extracted MedQA answer text. Each refusal example pairs an out-of-scope question with its
retrieved context and the target NOT_IN_CORPUS. The prompt is the strict grounding prompt used in
notebook 22, so training and inference share the same instruction.

In [ ]:
STRICT = ("You are an expert medical AI. Read the German clinical guidelines and answer the "
          "medical question in ENGLISH. Use ONLY the provided German context. If the context does "
          "not contain the answer, reply with exactly: NOT_IN_CORPUS")

def user_msg(context, question):
    return f"{STRICT}\n\nContext (German):\n{context}\n\nQuestion (English):\n{question}\n\nAnswer (English):"

records = []
for _, r in df_pos.iterrows():
    ctx = "\n\n".join(hybrid_retrieve(str(r[Q_DE])))
    records.append({"user": user_msg(ctx, str(r[Q_EN])), "target": str(r["English_Correct_Text"]).strip()})

neg_n = min(len(df_neg), int(CONFIG["NEG_FRACTION"] * len(df_pos)))
for _, r in df_neg.sample(neg_n, random_state=42).iterrows():
    ctx = "\n\n".join(hybrid_retrieve(str(r[Q_DE])))
    records.append({"user": user_msg(ctx, str(r[Q_EN])), "target": "NOT_IN_CORPUS"})

print("training examples:", len(records), "(answerable:", len(df_pos), "| refusals:", neg_n, ")")

## 6. Load Mistral-7B in 4-bit and attach LoRA adapters

The LoRA update matrices are zero-initialised, so immediately after this cell the adapted model
behaves identically to the base model. Section 8 uses this property to measure the base scores on
the same object before any training step runs.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = CONFIG["MODEL"], max_seq_length = CONFIG["MAX_SEQ"], load_in_4bit = True, dtype = None)
model = FastLanguageModel.get_peft_model(
    model, r = 16, lora_alpha = 16, lora_dropout = 0.0,
    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing = "unsloth", random_state = 42)

## 7. Evaluation harness

Answerability on the evaluation set is taken from the independent corpus-grounded ground truth of
notebook 22: a question is answerable when its corpus_ground_truth is not NOT_IN_CORPUS. Two
language-agnostic proxies are reported, since a lexical grounding check is unreliable between an
English answer and a German context: refusal recall on out-of-corpus questions, and over-refusal
on answerable questions. Grounding itself is measured by RAGAS faithfulness in Section 10. Each
evaluated question also records its answer and context for the RAGAS run.

In [ ]:
gt = pd.read_csv(find(CONFIG["GT_FILE"]))
ref_map = dict(zip(gt[CONFIG["GT_Q"]].astype(str).str.strip(), gt[CONFIG["GT_REF"]].astype(str)))

def answerable(q):
    r = ref_map.get(str(q).strip())
    return bool(r) and str(r).strip().upper() != "NOT_IN_CORPUS"

def generate(user_text, max_new_tokens=256):
    ids = tokenizer.apply_chat_template([{"role": "user", "content": user_text}],
                                        add_generation_prompt=True, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(input_ids=ids, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()

def evaluate(tag, n=CONFIG["EVAL_N"]):
    scored = eval200[eval200[Q_EN].astype(str).str.strip().isin(ref_map)].head(n)
    correct_refusal = over_refusal = n_unans = n_ans = 0
    rows = []
    for _, r in scored.iterrows():
        q, gq = str(r[Q_EN]), str(r[Q_DE])
        ctxs = hybrid_retrieve(gq)
        out  = generate(user_msg("\n\n".join(ctxs), q))
        refused = out.upper().startswith("NOT_IN_CORPUS")
        rows.append({"question": q, "answer": out, "contexts": ctxs, "ground_truth": ref_map.get(q.strip())})
        if answerable(q):
            n_ans += 1; over_refusal += int(refused)
        else:
            n_unans += 1; correct_refusal += int(refused)
    scores = {
        "model": tag,
        "refusal_recall (out-of-corpus)": round(correct_refusal / max(n_unans, 1), 3),
        "over_refusal (answerable)":      round(over_refusal / max(n_ans, 1), 3),
        "n_ans/n_unans": f"{n_ans}/{n_unans}",
    }
    return scores, rows

base_scores, base_rows = evaluate("base (pre-training)")
base_scores

## 8. Supervised fine-tuning

A short QLoRA run over the constructed examples. Two epochs is generally sufficient for a few
hundred samples; longer runs tend to overfit the refusal token. Depending on the installed trl
version, two SFTConfig fields (dataset_text_field, max_seq_length) may need to move into the
trainer call instead, as the comment marks.

In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

def to_text(r):
    msgs = [{"role": "user", "content": r["user"]}, {"role": "assistant", "content": r["target"]}]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False)}

train_ds = Dataset.from_list(records).map(to_text)

trainer = SFTTrainer(
    model = model, tokenizer = tokenizer, train_dataset = train_ds,
    args = SFTConfig(
        per_device_train_batch_size = 2, gradient_accumulation_steps = 4, warmup_steps = 5,
        num_train_epochs = CONFIG["EPOCHS"], learning_rate = 2e-4, logging_steps = 10,
        optim = "adamw_8bit", weight_decay = 0.01, lr_scheduler_type = "linear", seed = 42,
        report_to = "none", output_dir = CONFIG["OUTPUT_DIR"],
        dataset_text_field = "text",   # trl>=0.11: keep here; older trl: pass to SFTTrainer(...)
        max_seq_length = CONFIG["MAX_SEQ"],
    ),
)
trainer.train()

## 9. Re-score the fine-tuned model and compare

The same harness runs on the same evaluation questions with the same retrieved context. Only the
adapter weights differ.

In [ ]:
tuned_scores, tuned_rows = evaluate("fine-tuned")
comparison = pd.DataFrame([base_scores, tuned_scores])
comparison

## 10. RAGAS evaluation (comparable to Chapter 4)

The answers were produced locally by the self-hosted model, so RAGAS spends external calls only on
the judge that scores the collected question, context, and answer rows against the independent
corpus-grounded reference. This is a single scoring pass over the evaluation set. The judge is the
same one used for the reported numbers in notebook 22.

In [ ]:
os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

from datasets import Dataset, Features, Value, Sequence
from ragas import evaluate as ragas_evaluate
from ragas.metrics import context_precision, context_recall, faithfulness, answer_relevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI

judge = LangchainLLMWrapper(ChatOpenAI(
    model = CONFIG["JUDGE_MODEL"], api_key = os.environ["OPENROUTER_API_KEY"],
    base_url = "https://openrouter.ai/api/v1", temperature = 0, max_retries = 6))
judge_emb = LangchainEmbeddingsWrapper(bge)
feat = Features({"question": Value("string"), "answer": Value("string"),
                 "contexts": Sequence(Value("string")), "ground_truth": Value("string")})

def run_ragas(rows):
    rows = [r for r in rows if r.get("ground_truth") and str(r["ground_truth"]).strip().upper() != "NOT_IN_CORPUS"]
    ds = Dataset.from_dict({
        "question":     [r["question"]     for r in rows],
        "answer":       [r["answer"]       for r in rows],
        "contexts":     [r["contexts"]     for r in rows],
        "ground_truth": [r["ground_truth"] for r in rows],
    }, features=feat)
    out = ragas_evaluate(ds, metrics=[context_precision, context_recall, faithfulness, answer_relevancy],
                         llm=judge, embeddings=judge_emb)
    return out.to_pandas()[["context_precision","context_recall","faithfulness","answer_relevancy"]].mean().round(3)

print("base model RAGAS:")
print(run_ragas(base_rows))
print("fine-tuned model RAGAS:")
print(run_ragas(tuned_rows))

In [ ]:
# Persist the adapter so it can be reloaded or served later.
model.save_pretrained(f"{CONFIG['OUTPUT_DIR']}/lora")
tokenizer.save_pretrained(f"{CONFIG['OUTPUT_DIR']}/lora")
print("saved to", f"{CONFIG['OUTPUT_DIR']}/lora")

## 11. Interpretation

A favourable outcome is a higher refusal recall on out-of-corpus questions together with a stable
or improved RAGAS faithfulness, without a rise in over-refusal on answerable questions. If refusal
recall and over-refusal rise together, the model has only learned to refuse more often, which is
not the objective.

Two limitations qualify the result. The training set is small, so the adapter can overfit the
refusal token, which makes the over-refusal column the primary diagnostic. The disease-scope proxy
used to label training answerability is coarser than a per-answer corpus check. The comparison that
matches Chapter 4 places this adapter in the generation stage of the full pipeline, with the
notebook 22 retrieval and the strong judge; the RAGAS row above is that measurement on the
evaluation subset.